# Common exclusion conventions in the walk corpus

This notebook presents the annotation and package-name inventory extracted by `scan_conventions.py`. The scan uses each library's configured end-of-window source snapshot and strips comments and literals before counting.

The recorded run covers **10,372 Java files**, **29 libraries** in **28 repositories**, **224 annotation names**, **681 packages**, and **447 package components**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

HERE = Path.cwd()
if not (HERE / 'data').is_dir():
    HERE = Path('results/exclusions')
DATA = HERE / 'data'

summary = pd.read_csv(DATA / 'summary.csv').iloc[0]
annotations = pd.read_csv(DATA / 'annotations.csv')
components = pd.read_csv(DATA / 'package-components.csv')
packages = pd.read_csv(DATA / 'packages.csv')

display(summary.to_frame('value'))

## Annotation findings

The table reports the configured common exclusions. `InternalApi` (17 uses in 17 Log4j API files) and `ExperimentalApi` (28 uses in 25 protobuf files) were discovered by the inventory and added because their names and project documentation explicitly identify unstable or non-public API. `PackagePrivate` is retained from the original configuration even though no real use remains in the end snapshot.

In [ ]:
excluded_annotations = [
    'Internal', 'InternalApi', 'Beta', 'Experimental', 'ExperimentalApi',
    'Test', 'VisibleForTesting', 'DoNotCall', 'RestrictedApi', 'PackagePrivate',
]
annotation_evidence = (
    pd.DataFrame({'annotation': excluded_annotations})
    .merge(annotations, on='annotation', how='left')
    .fillna({'occurrences': 0, 'files': 0, 'libraries': 0, 'library_ids': ''})
)
display(annotation_evidence[['annotation', 'occurrences', 'files', 'libraries', 'library_ids']])

In [ ]:
top = annotations.nlargest(15, 'occurrences').sort_values('occurrences')
ax = top.plot.barh(x='annotation', y='occurrences', legend=False, figsize=(8, 5), logx=True)
ax.set(title='Most-used annotations (log scale)', xlabel='Occurrences', ylabel='')
plt.tight_layout()

## Package-name findings

Exact dotted package components provide useful common exclusions. `internal`, `impl`, and `experimental` collectively occur in 863 files from 97 packages across 15 libraries. Test-oriented components (`test`, `testbed`, and `testsuite`) identify another 121 files from broad historical source roots. Components such as `util`, `support`, and `spi` are deliberately not excluded because they frequently contain supported public API.

In [ ]:
excluded_components = [
    'internal', 'impl', 'experimental', 'example', 'examples',
    'test', 'tests', 'testbed', 'testsuite',
]
component_evidence = (
    pd.DataFrame({'component': excluded_components})
    .merge(components, on='component', how='left')
    .fillna({'packages': 0, 'files': 0, 'libraries': 0, 'library_ids': ''})
)
display(component_evidence[['component', 'packages', 'files', 'libraries', 'library_ids']])

plotted = component_evidence.query('files > 0').sort_values('files')
ax = plotted.plot.barh(x='component', y='files', legend=False, figsize=(8, 4))
ax.set(title='Files under excluded package components', xlabel='Java files', ylabel='')
plt.tight_layout()

## Interpretation

The results support a single convention-based configuration across all libraries. Simple annotation names intentionally cover equivalent annotations from different packages. Package exclusions use exact dotted components to avoid accidental substring matches. This is a cross-sectional validation at the end of each analysis window, not an assertion that the same conventions existed at every historical commit.